**Live Code 3**

Nama  : Baihaqsani

Batch : FTDS-045


> Mengidentifikasi data harga penyewaan properti yang bersifat anomaly atau outlier ini, untuk manentukan campaign marketing selanjutnya.



1. Pada tugas kali ini, dataset yang akan Anda gunakan berasal dari Google BigQuery. Gunakan informasi dibawah ini sebagai tempat untuk mengambil dataset.

Project ID : bigquery-public-data

Dataset : properati_properties_ar

Tabel : properties_rent_201501


In [1]:
df = client.query('''
SELECT price,
FROM `bigquery-public-data.properati_properties_ar.properties_rent_201501`
WHERE operation = 'rent' AND property_type = 'apartment'
LIMIT 5000;
''').to_dataframe()

df

NameError: name 'client' is not defined

2. Anda dapat mengkoneksikan notebook pada Google Colab dengan Google BigQuery dengan Python code berikut :

In [2]:
from google.colab import auth
from google.cloud import bigquery
!gcloud auth application-default login
print('Authenticated')

GCP_PROJECT_ID = 'baihaqsani-hactiv8' # Use your GCP Project ID
client = bigquery.Client(project = GCP_PROJECT_ID)

ModuleNotFoundError: No module named 'google'

**PROBLEM**

Anda adalah seorang Data Analyst pada sebuah perusahaan penyewaan properti. Tim marketing perusahaan Anda ingin melakukan sebuah campaign promo terhadap harga penyewaan properti dimana mereka membutuhkan data harga dengan tidak adanya outlier pada data tersebut. Anda diberi tugas untuk mengidentifikasi data harga penyewaan properti yang bersifat anomaly atau outlier ini.

Section 1 - Anomaly Analysis


Untuk melakukan pengecekan data anomaly atau data outlier, lakukanlah beberapa langkah dibawah ini :

1. Lakukan perhitungan Central Tendency (Mean, Median, dan Modus) terhadap data yang telah Anda peroleh.

2. Cek nilai skewness data untuk mengetahui apakah data terdistribusi normal atau tidak.

3. Lakukan pengolahan data dengan menggunakan Extreme Value Analysis. Nyatakan secara jelas berapa banyak persentase outlier yang Anda temukan.

4. Buatlah sebuah variabel baru yang menyimpan data bersih yang sudah tidak memiliki data outlier.

5. Simpan data yang sudah bersih ini ke dalam sebuah file CSV dengan format P0LC3_<nama-student>_data_clean.csv.

In [37]:

# Central Tendency (Mean, Median, Modus)
mean_val = df['price'].mean()
median_val = df['price'].median()
mode_val = df['price'].mode()[0]

# Cek nilai skewness data
skew_val = df['price'].skew()
kurt_val = df['price'].kurtosis()

print(f"Mean : {mean_val:.2f}")
print(f"Median : {median_val:.2f}")
print(f"Modus : {mode_val:.2f}")

print(f"Skewness : {skew_val:.2f}")
print(f"Kurtosis : {kurt_val:.2f}")


Mean : 5166.36
Median : 2700.00
Modus : 0.00
Skewness : 4.24
Kurtosis : 25.48


Dari program diatas didapatkan nilai :
Mean 5166.36 ,Median 2700 ,Modus 0 ,Skewness 4.24, Kurtosis 25.48.


In [ ]:
from ast import Index
# Pengolahan nilai ekstrem menggunakan Metode IQR
Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)
IQR = Q3-Q1

# Menentukan Upper dan Lower Bound
upper_bound = Q3 + 1.5 * IQR
lower_bound = Q1 - 1.5 * IQR

# Filter data outlier dan hitung persentasenya
outliers = df[(df['price'] < lower_bound) | (df['price'] > upper_bound)]
persentase_outlier = (len(outliers) / len(df)) * 100

print(f"Total bari outlier:{len(outliers)} baris.")
print(f"Persentase Outlier: {persentase_outlier:.2f}%\n")

# Membersihkan dataframe dari outlier
df_clean = df[(df['price'] >= lower_bound) & (df['price'] <= upper_bound)]

# Perbandingan
print(f"Sebelum dibersihkan : Mean: {mean_val:.2f} | Median: {median_val:.2f}")
print(f"Sesudah dibersihkan : Mean: {df_clean['price'].mean():.2f} | Median : {df_clean['price'].median():.2f}")

# Menyimpan data bersih ke bentuk CSV
df_clean.to_csv("P0LC3_Baihaqsani_data_clean.csv", index=False)
print("\nFile 'P0LC3_Baihaqsani_data_clean.csv' berhasil disimpan ")


Total bari outlier:158 baris.
Persentase Outlier: 9.53%

Sebelum dibersihkan : Mean: 5166.36 | Median: 2700.00
Sesudah dibersihkan : Mean: 2875.08 | Median : 2500.00

File 'P0LC3__data_clean.csv' berhasil disimpan 


Dari pogram diatas terdapat 158 baris yang dideteksi sebagai outlier. Terlihat tampilan data setelah dilakukan pengolahan data mengunnakan perhitungan IQR.

**Section 2 - API**

Buatlah sebuah Python file yang berisi script API sederhana menggunakan FastAPI untuk menampilkan data yang sudah bersih (data yang sudah tidak ada lagi anomalinya).

Petunjuk :

Load data CSV yang sudah Anda simpan sebelumnya.
Lakukan konversi tipe data ke dalam tipe data dictionary. Anda dapat menggunakan sintaks df.to_dict() atau df.to_json() agar dapat ditampilkan dengan API menggunakan FastAPI.
Simpan Python file ini dengan format P0LC3_<nama-student>_app.py.

In [40]:
from fastapi import FastAPI, HTTPException
import pandas as pd

app = FastAPI()

# Mencoba load data CSV dan mengkonversi nya ke dictionary saat API pertama kali di jalankan
try:
  df = pd.read_csv('P0LC3__data_clean.csv')
  # Orient="records" akan mengubah tabel menjadi list berisi dictionary
  data_clean = df.to_dict(orient="records")
except FileNotFoundError:
  data_clean = []


@app.get("/")
def home():
  return {"message" : "API data clean"}

# Menampilkan seluruh entry data
@app.get("/data")
def mendapatkan_semua_data():
  return{
      "total_data" : len(data_clean),
      "data": data_clean
  }

Data diatas menjukkan cara mengkonversi tipe data ke dalam dictionary dengan menggunakan sintaks 'df.to_dict' dan menampilkan dengan menggunakan FastAPI.

**Section 3 - Data Analysis**

Jawablah pertanyaan-pertanyaan dibawah ini berdasarkan hasil yang Anda peroleh sebelumnya. Anda dapat menggunakan Markdown untuk memperjelas jawaban Anda.

1. Berapa nilai rata-rata, median, dan modus dari data yang Anda peroleh sebelum dihilangkan outliernya ? Bagaimana kecenderungan pemusatan datanya ? Jelaskan analisa Anda !

2. Berdasarkan nilai skewness yang Anda peroleh, bagaimana dengan distribusi datanya? Jelaskan analisa Anda !

3. Ada dua teknik yang dapat dipakai untuk melakukan Extreme Value Analysis. Teknik manakah yang Anda pakai ? Berikan alasan Anda memakainya dengan berdasarkan data !

**JAWABAN SECTION 3**
1. Nilai yang didapatkan sebelum dihilangkan outlier nya adalah sebagai berikut :
*   Mean = 5166.36
*   Median = 2700.00
*   Modus = 0.00


Berdasarkan analisa sebelum dihilangkan outlier nya, pemusatan data nya memiliki kecenderungan di harga tinggi. Karena hasil Mean menunjukkan nilai yang jauh lebih tinggi dibandingkan dengan nilai Mediannya.

2. Berdasarkan dari nilai skewness yang ditampilkan yaitu sebesar 4.24, menunjukkan bahwa ekor kurva lebih condong dan memanjang ke kanan yang dimana nilai Mean > Median. Hal tersebut menunjukkan bahwa nilai outliers yang masih sangat tinggi.

3. Saya menggunakan teknik IQR pada Extreme Value Analysis. Alasan saya menggunakan teknik IQR adalah memudahkan dalam proses pengolahan data untuk menghilangkan nilai outliers dari data tersebut. Penggunaan teknik IQR ini terlihat sangat efektif, terlihat dari hasil sesudah dibersihkan :
Mean: 2875.08
Median : 2500.00

dimana nilai mean berubah sangat signifikan dari yang sebelumnya 5166.36 menjadi 2875.08, hal itu menunjukkan bawah outlier banyak terdapat di harga yang tingi sehingga menimbulkan nilai Mean yang terlalu tinggi.


